# Préparation des données Usagers 2020–2024

Ce notebook est consacré à la préparation, à l'évaluation de la qualité et au nettoyage des données relatives aux usagers impliqués dans les accidents corporels de la circulation entre 2020 et 2024.

L'objectif est d'obtenir une table consolidée et nettoyée, exploitable dans les prochaines étapes du projet, notamment pour l'analyse de la gravité des accidents.

## 1. Préparation et consolidation des données

In [460]:
# 1.1. Chargement des fichiers annuels

import pandas as pd
import os
dfs_usagers = {}

for annee in range(2020, 2025):
    chemin = f"data/raw/{annee}/usagers-{annee}.csv"
    
    dfs_usagers[annee] = pd.read_csv(
        chemin,
        sep=";"
    )

    print(f"{annee} : {dfs_usagers[annee].shape}")

2020 : (105295, 15)
2021 : (129248, 16)
2022 : (126662, 16)
2023 : (125789, 16)
2024 : (125187, 16)


### 1.2. Comparaison de la structure des fichiers

La structure des fichiers annuels est comparée avant leur consolidation afin d'identifier les éventuelles différences de colonnes entre les années.

Cette étape permet également d'évaluer le rôle des variables et leur intérêt pour l'analyse de la gravité des accidents, afin de définir les variables à conserver avant la consolidation.

In [461]:
# 1.2. Comparaison de la structure des fichiers

for annee, dataframe in dfs_usagers.items():
    print(f"\n{annee}")
    print("Colonnes :", dataframe.columns.tolist())
    print("Types :")
    print(dataframe.dtypes)


2020
Colonnes : ['Num_Acc', 'id_vehicule', 'num_veh', 'place', 'catu', 'grav', 'sexe', 'an_nais', 'trajet', 'secu1', 'secu2', 'secu3', 'locp', 'actp', 'etatp']
Types :
Num_Acc        int64
id_vehicule      str
num_veh          str
place          int64
catu           int64
grav           int64
sexe           int64
an_nais        int64
trajet         int64
secu1          int64
secu2          int64
secu3          int64
locp           int64
actp             str
etatp          int64
dtype: object

2021
Colonnes : ['Num_Acc', 'id_usager', 'id_vehicule', 'num_veh', 'place', 'catu', 'grav', 'sexe', 'an_nais', 'trajet', 'secu1', 'secu2', 'secu3', 'locp', 'actp', 'etatp']
Types :
Num_Acc          int64
id_usager          str
id_vehicule        str
num_veh            str
place            int64
catu             int64
grav             int64
sexe             int64
an_nais        float64
trajet           int64
secu1            int64
secu2            int64
secu3            int64
locp             int6

### 1.3. Décisions d'harmonisation avant consolidation

#### Décision concernant `id_usager`

La variable `id_usager` est disponible dans les fichiers BAAC de 2021 à 2024 mais absente du fichier 2020.

Afin de préserver l'identifiant fourni par la source officielle, `id_usager` est conservée. Pour l'année 2020, la colonne sera créée avec une valeur manquante (`NaN`) afin d'harmoniser la structure des cinq fichiers.

Aucun identifiant artificiel ne sera créé à cette étape.

Lors de la modélisation relationnelle, un identifiant technique propre au projet pourra être créé pour l'ensemble des usagers 2020-2024 afin de disposer d'une clé primaire unique, tout en conservant `id_usager` comme identifiant source lorsqu'il est disponible.

In [462]:
# 1.3.1. Harmonisation de id_usager pour 2020

dfs_usagers[2020]["id_usager"] = pd.NA

#### Décision concernant `num_veh`

La variable `num_veh` correspond au numéro du véhicule attribué dans l'accident.

L'analyse réalisée précédemment dans la table Véhicules a montré que cette variable n'est pas suffisamment fiable comme identifiant et qu'elle est redondante avec `id_vehicule`.

La variable `num_veh` ne sera donc pas conservée. `id_vehicule`, disponible sur l'ensemble de la période 2020-2024, sera utilisée pour assurer la relation entre les tables Usagers et Véhicules.

In [463]:
# 1.4. Analyse de la variable place

for annee, dataframe in dfs_usagers.items():
    print(f"\n{annee}")
    print(dataframe["place"].value_counts(dropna=False).sort_index())


2020
place
1     78168
2     12006
3      1980
4      1787
5       523
6       136
7       968
8       446
9      1038
10     8243
Name: count, dtype: int64

2021
place
-1        17
 1     97026
 2     14473
 3      2475
 4      2230
 5       620
 6       147
 7      1106
 8       448
 9      1189
 10     9517
Name: count, dtype: int64

2022
place
-1         8
 1     94292
 2     14001
 3      2459
 4      2232
 5       728
 6       184
 7      1241
 8       610
 9      1340
 10     9567
Name: count, dtype: int64

2023
place
-1         2
 1     93440
 2     14060
 3      2420
 4      2271
 5       764
 6       202
 7      1218
 8       583
 9      1308
 10     9521
Name: count, dtype: int64

2024
place
-1         3
 1     92567
 2     14120
 3      2500
 4      2419
 5       703
 6       184
 7      1304
 8       625
 9      1361
 10     9401
Name: count, dtype: int64


#### Décision concernant `place`

La variable `place` présente des modalités cohérentes sur l'ensemble de la période 2020-2024.

Quelques valeurs `-1`, correspondant à des informations non renseignées, sont observées à partir de 2021. Elles seront remplacées par des valeurs manquantes lors du nettoyage.

La position occupée par l'usager dans le véhicule pouvant être associée à la gravité, la variable `place` est conservée.

In [464]:
# 1.5. Analyse de la variable catu

for annee, dataframe in dfs_usagers.items():
    print(f"\n{annee}")
    print(dataframe["catu"].value_counts(dropna=False).sort_index())


2020
catu
1    78173
2    18879
3     8243
Name: count, dtype: int64

2021
catu
1    97276
2    22455
3     9517
Name: count, dtype: int64

2022
catu
1    94421
2    22675
3     9566
Name: count, dtype: int64

2023
catu
1    93462
2    22806
3     9521
Name: count, dtype: int64

2024
catu
1    92581
2    23205
3     9401
Name: count, dtype: int64


#### Décision concernant `catu`

La variable `catu` présente les trois catégories d'usagers attendues de manière cohérente sur l'ensemble de la période 2020-2024 : conducteur, passager et piéton.

Aucune valeur `-1` ou modalité inattendue n'est observée.

Cette variable est directement pertinente pour l'analyse des profils associés à la gravité des accidents. Elle est donc conservée.

In [465]:
# 1.6. Analyse de la variable grav

for annee, dataframe in dfs_usagers.items():
    print(f"\n{annee}")
    print(dataframe["grav"].value_counts(dropna=False).sort_index())


2020
grav
1    43267
2     2780
3    16775
4    42473
Name: count, dtype: int64

2021
grav
-1       60
 1    55143
 2     3219
 3    19093
 4    51733
Name: count, dtype: int64

2022
grav
-1      241
 1    53630
 2     3550
 3    19260
 4    49981
Name: count, dtype: int64

2023
grav
-1      118
 1    53399
 2     3398
 3    19271
 4    49603
Name: count, dtype: int64

2024
grav
1    52920
2     3432
3    19126
4    49709
Name: count, dtype: int64


#### Décision concernant `grav`

La variable `grav` présente les quatre catégories de gravité attendues sur l'ensemble de la période 2020-2024 : indemne, tué, blessé hospitalisé et blessé léger.

Quelques valeurs `-1`, correspondant à des informations non renseignées, sont observées entre 2021 et 2023. Elles seront remplacées par des valeurs manquantes lors du nettoyage.

La variable `grav` est centrale pour l'étude de la gravité des accidents et pourra notamment servir de variable cible lors de la modélisation prédictive. Elle est donc conservée.

Les codes de `grav` représentent des catégories et ne doivent pas être interprétés comme une échelle numérique ordonnée.

In [466]:
# 1.7. Analyse de la variable sexe

for annee, dataframe in dfs_usagers.items():
    print(f"\n{annee}")
    print(dataframe["sexe"].value_counts(dropna=False).sort_index())


2020
sexe
1    72458
2    32837
Name: count, dtype: int64

2021
sexe
-1     3062
 1    86235
 2    39951
Name: count, dtype: int64

2022
sexe
-1     2744
 1    84795
 2    39123
Name: count, dtype: int64

2023
sexe
-1     2431
 1    84013
 2    39345
Name: count, dtype: int64

2024
sexe
-1     2395
 1    83864
 2    38928
Name: count, dtype: int64


#### Décision concernant `sexe`

La variable `sexe` présente les modalités renseignées attendues sur l'ensemble de la période.

Des valeurs `-1`, correspondant à des informations non renseignées, apparaissent à partir de 2021. Elles seront remplacées par des valeurs manquantes lors du nettoyage, sans imputation à ce stade.

Cette variable étant pertinente pour l'analyse des profils d'usagers associés à la gravité, elle est conservée.

In [467]:
# 1.8. Analyse de la variable an_nais

for annee, dataframe in dfs_usagers.items():
    print(
        f"{annee} : "
        f"manquantes = {dataframe['an_nais'].isna().sum()} | "
        f"min = {dataframe['an_nais'].min()} | "
        f"max = {dataframe['an_nais'].max()}"
    )

2020 : manquantes = 0 | min = 1900 | max = 2020
2021 : manquantes = 3067 | min = 1912.0 | max = 2021.0
2022 : manquantes = 2874 | min = 1913.0 | max = 2022.0
2023 : manquantes = 2598 | min = 1913.0 | max = 2023.0
2024 : manquantes = 2579 | min = 1914.0 | max = 2024.0


#### Décision concernant `an_nais`

La variable `an_nais` présente des années de naissance cohérentes avec les années d'accident sur l'ensemble de la période 2020-2024.

Des valeurs manquantes sont présentes à partir de 2021, ce qui explique le type `float64` observé pour ces années.

La variable est conservée car elle permettra notamment de calculer l'âge de l'usager au moment de l'accident. Elle sera convertie en type entier nullable (`Int64`) lors du nettoyage.

In [468]:
# 1.9. Analyse de la variable trajet

for annee, dataframe in dfs_usagers.items():
    print(f"\n{annee}")
    print(dataframe["trajet"].value_counts(dropna=False).sort_index())


2020
trajet
-1      213
 0    26793
 1    14481
 2     2022
 3     4255
 4     9774
 5    38291
 9     9466
Name: count, dtype: int64

2021
trajet
-1     3270
 0    32567
 1    17337
 2     2864
 3     4532
 4    12183
 5    45457
 9    11038
Name: count, dtype: int64

2022
trajet
-1     2874
 0    32737
 1    16282
 2     2687
 3     3713
 4    11622
 5    46253
 9    10494
Name: count, dtype: int64

2023
trajet
-1     2499
 0    33303
 1    16475
 2     2986
 3     3598
 4    11003
 5    46131
 9     9794
Name: count, dtype: int64

2024
trajet
-1     2626
 0    35024
 1    15582
 2     2828
 3     3685
 4    10589
 5    45375
 9     9478
Name: count, dtype: int64


#### Décision concernant `trajet`

La variable `trajet`, correspondant au motif du déplacement, présente des modalités cohérentes sur l'ensemble de la période 2020-2024.

Les codes `-1` et `0` correspondent à des informations non renseignées. Ils seront remplacés par des valeurs manquantes lors du nettoyage, sans imputation dans le fichier de référence.

Les autres modalités renseignent différents motifs de déplacement, tels que les trajets domicile-travail, domicile-école, professionnels ou de loisirs.

Cette variable pouvant contribuer à l'analyse des contextes de déplacement associés à la gravité, elle est conservée.

In [469]:
# 1.10. Analyse des variables de sécurité

for variable in ["secu1", "secu2", "secu3"]:
    print(f"\n--- {variable} ---")

    for annee, dataframe in dfs_usagers.items():
        print(f"\n{annee}")
        print(dataframe[variable].value_counts(dropna=False).sort_index())


--- secu1 ---

2020
secu1
-1       33
 0     9962
 1    62109
 2    20328
 3      632
 4       64
 5       26
 6       79
 7        4
 8    11945
 9      113
Name: count, dtype: int64

2021
secu1
-1     2791
 0    11880
 1    75720
 2    23946
 3      821
 4       75
 5       24
 6      114
 7        3
 8    13765
 9      109
Name: count, dtype: int64

2022
secu1
-1     2679
 0    12144
 1    73953
 2    22651
 3      774
 4      102
 5       25
 6       97
 7        5
 8    14117
 9      115
Name: count, dtype: int64

2023
secu1
-1     2292
 0    11938
 1    72988
 2    22012
 3      726
 4       67
 5       27
 6      135
 7        1
 8    15480
 9      123
Name: count, dtype: int64

2024
secu1
-1     2103
 0    12216
 1    72919
 2    21246
 3      723
 4      104
 5       25
 6      126
 7        2
 8    15584
 9      139
Name: count, dtype: int64

--- secu2 ---

2020
secu2
-1    38656
 0    41857
 1      221
 2      269
 3      116
 4      910
 5     1188
 6     9687
 7      142


#### Décision concernant les équipements de sécurité

La variable `secu1` décrit l'équipement de sécurité principal de l'usager. Elle est suffisamment renseignée sur l'ensemble de la période et présente un intérêt direct pour l'étude de la gravité. Elle est conservée.

La variable `secu2`, correspondant à un équipement de sécurité complémentaire, présente 262 577 codes `-1`, soit environ 42,9 % des observations. Compte tenu de son caractère complémentaire et de son taux élevé de non-renseignement, elle ne sera pas retenue dans le jeu de données final.

La variable `secu3`, encore plus faiblement renseignée sur la période étudiée, n'est également pas conservée.

Le jeu de données final conservera donc uniquement `secu1` pour représenter l'information relative aux équipements de sécurité.

In [470]:
# 1.11. Analyse des variables spécifiques aux piétons

for variable in ["locp", "actp", "etatp"]:
    print(f"\n--- {variable} ---")

    for annee, dataframe in dfs_usagers.items():
        print(f"\n{annee}")
        print(dataframe[variable].value_counts(dropna=False).sort_index())


--- locp ---

2020
locp
-1    50568
 0    46673
 1     1199
 2     1826
 3     2445
 4     1240
 5      628
 6      229
 7        7
 8      100
 9      380
Name: count, dtype: int64

2021
locp
-1    64620
 0    55299
 1     1346
 2     2129
 3     2850
 4     1409
 5      767
 6      262
 7       12
 8      139
 9      415
Name: count, dtype: int64

2022
locp
-1    51092
 0    66113
 1     1364
 2     1989
 3     3059
 4     1388
 5      740
 6      298
 7        7
 8      138
 9      474
Name: count, dtype: int64

2023
locp
-1    63330
 0    52354
 1     1455
 2     1978
 3     3271
 4     1529
 5      842
 6      336
 7       11
 8      152
 9      531
Name: count, dtype: int64

2024
locp
-1    61771
 0    53151
 1     1457
 2     2063
 3     3365
 4     1539
 5      847
 6      330
 7        7
 8      157
 9      500
Name: count, dtype: int64

--- actp ---

2020
actp
 -1    46593
0      50627
1        523
2        252
3       5926
4        128
5        296
6         28
7         11

#### Décision concernant les variables spécifiques aux piétons

Les variables `locp`, `actp` et `etatp` décrivent des caractéristiques spécifiques aux piétons.

Leur forte proportion de valeurs `-1` ou `0` sur l'ensemble des usagers s'explique notamment par leur caractère non applicable aux conducteurs et aux passagers. Elles ne doivent donc pas être considérées uniquement comme des variables de mauvaise qualité.

Cependant, le projet vise principalement une analyse globale des facteurs associés à la gravité pour l'ensemble des usagers. Ces variables très spécifiques aux piétons complexifieraient le jeu de données global et seraient peu exploitables pour la majorité des observations.

Elles ne sont donc pas conservées dans le jeu de données principal. La variable `catu` est conservée et permettra d'identifier les piétons dans les analyses.

In [471]:
# 1.12. Sélection des variables

colonnes_usagers = [
    "Num_Acc",
    "id_usager",
    "id_vehicule",
    "place",
    "catu",
    "grav",
    "sexe",
    "an_nais",
    "trajet",
    "secu1"
]

dfs_usagers_selection = {
    annee: dataframe[colonnes_usagers].copy()
    for annee, dataframe in dfs_usagers.items()
}

for annee, dataframe in dfs_usagers_selection.items():
    print(f"{annee} : {dataframe.shape}")

2020 : (105295, 10)
2021 : (129248, 10)
2022 : (126662, 10)
2023 : (125789, 10)
2024 : (125187, 10)


#### Résultat

Après sélection, les cinq fichiers annuels présentent une structure homogène de 10 variables.

La variable `id_usager` est conservée afin de préserver l'identifiant fourni par la source BAAC lorsqu'il est disponible. Pour 2020, cette variable contient des valeurs manquantes.

La variable `secu1` est conservée pour représenter l'équipement de sécurité principal, tandis que `secu2` et `secu3` ne sont pas retenues.

Les variables sélectionnées sont désormais compatibles entre les cinq années et peuvent être consolidées.

In [472]:
# 1.13. Consolidation des fichiers annuels

df_usagers_concat = pd.concat(
    dfs_usagers_selection.values(),
    ignore_index=True
)

print("Dimensions :", df_usagers_concat.shape)

Dimensions : (612181, 10)


In [473]:
# 1.14. Sauvegarde des données consolidées

df_usagers_concat.to_csv(
    "data/interim/usagers_2020_2024.csv",
    sep=";",
    index=False
)

print("Fichier intermédiaire sauvegardé.")

Fichier intermédiaire sauvegardé.


#### Résultat

Les fichiers Usagers de 2020 à 2024 ont été consolidés dans un seul DataFrame contenant 612 181 lignes et 10 variables.

La structure est homogène sur l'ensemble de la période et aucune ligne n'a été supprimée lors de la consolidation.

La variable `id_usager` est conservée afin de préserver l'identifiant fourni par la source BAAC lorsqu'il est disponible. Pour l'année 2020, cette variable contient des valeurs manquantes.

La variable `secu1` est conservée pour représenter l'équipement de sécurité principal, tandis que `secu2` et `secu3` ne sont pas retenues dans le jeu de données consolidé.

## 2. Contrôle qualité des données consolidées

In [474]:
# 2.1. Contrôle des doublons complets

print("Doublons complets :", df_usagers_concat.duplicated().sum())

Doublons complets : 67


In [475]:
# 2.2. Analyse des doublons selon id_usager

doublons_usagers = df_usagers_concat[
    df_usagers_concat.duplicated(keep=False)
]

print("Lignes concernées :", len(doublons_usagers))
print()
print(
    doublons_usagers["id_usager"]
    .isna()
    .value_counts()
)

Lignes concernées : 119

id_usager
True    119
Name: count, dtype: int64


#### Décision concernant les doublons

Le contrôle identifie 65 doublons complets, représentant 115 lignes concernées.

L'ensemble de ces lignes appartient à l'année 2020, pour laquelle `id_usager` n'est pas disponible dans les données sources. Aucun doublon complet n'est observé de 2021 à 2024 lorsque l'identifiant officiel `id_usager` est pris en compte.

En l'absence d'identifiant individuel permettant de distinguer les usagers en 2020, ces lignes ne peuvent pas être considérées avec certitude comme des doublons erronés.

Afin d'éviter la suppression potentielle d'usagers distincts, aucune ligne n'est supprimée. Un identifiant technique unique sera créé ultérieurement pour assurer l'identification de chaque enregistrement dans la base du projet.

In [476]:
# 2.3. Contrôle des valeurs manquantes

valeurs_manquantes = df_usagers_concat.isna().sum()

print(valeurs_manquantes)

Num_Acc             0
id_usager      105295
id_vehicule         0
place               0
catu                0
grav                0
sexe                0
an_nais         11118
trajet              0
secu1               0
dtype: int64


#### Résultat

Les valeurs manquantes détectées concernent principalement deux variables :

- `id_usager` : 105 295 valeurs manquantes, correspondant à l'année 2020 pour laquelle cet identifiant n'est pas disponible dans la source BAAC ;
- `an_nais` : 11 118 valeurs manquantes sur la période 2021-2024.

Les autres variables ne présentent pas de valeurs `NaN` à ce stade. Cependant, certaines utilisent des codes spécifiques pour représenter une information non renseignée. Ces codes doivent être contrôlés selon la définition de chaque variable avant le nettoyage.

In [477]:
# 2.4. Contrôle des codes -1

colonnes_codes = [
    "place",
    "catu",
    "grav",
    "sexe",
    "trajet",
    "secu1"
]

for colonne in colonnes_codes:
    print(f"{colonne} : {(df_usagers_concat[colonne] == -1).sum()}")

place : 30
catu : 0
grav : 419
sexe : 10632
trajet : 11482
secu1 : 9898


#### Résultat

Des codes `-1`, correspondant à des informations non renseignées, sont présents dans plusieurs variables.

Ils concernent `place`, `grav`, `sexe`, `trajet` et `secu1`. La variable `catu` ne présente aucun code `-1`.

Ces codes seront remplacés par des valeurs manquantes lors du nettoyage, sans imputation dans le fichier de référence.

In [478]:
# 2.5. Contrôle des codes 0

colonnes_zero = [
    "trajet",
    "secu1"
]

for colonne in colonnes_zero:
    print(f"{colonne} : {(df_usagers_concat[colonne] == 0).sum()}")

trajet : 160424
secu1 : 58140


#### Résultat

La variable `trajet` contient 160 424 codes `0` et 11 482 codes `-1`, soit 171 906 valeurs non renseignées au total (environ 28,1 % des observations).

Malgré cette proportion, environ 71,9 % des observations restent renseignées. La variable présente également un intérêt métier pour l'analyse des circonstances associées à la gravité des accidents. Elle est donc conservée.

Les codes `0` et `-1` de `trajet` seront remplacés par des valeurs manquantes lors du nettoyage.

Pour `secu1`, le code `0` constitue une modalité valide et sera conservé. Seul le code `-1` sera transformé en valeur manquante.

In [479]:
# 2.6. Contrôle des identifiants

for colonne in ["Num_Acc", "id_vehicule", "id_usager"]:
    print(
        f"{colonne} : "
        f"{df_usagers_concat[colonne].isna().sum()} valeurs manquantes"
    )

Num_Acc : 0 valeurs manquantes
id_vehicule : 0 valeurs manquantes
id_usager : 105295 valeurs manquantes


#### Résultat

Les identifiants `Num_Acc` et `id_vehicule` sont renseignés pour l'ensemble des 612 181 usagers.

La variable `id_usager` présente 105 295 valeurs manquantes, correspondant aux données de 2020 pour lesquelles cet identifiant n'est pas disponible dans la source BAAC.

Cette absence est donc liée à la structure des données sources et non à une anomalie de chargement.

In [480]:
# 2.7. Contrôle de l'intégrité des relations

df_caract_clean = pd.read_csv(
    "data/processed/caract_2020_2024_clean.csv",
    sep=";",
    dtype={"Num_Acc": str}
)

df_vehicules_clean = pd.read_csv(
    "data/processed/vehicules_2020_2024_clean.csv",
    sep=";",
    dtype={"Num_Acc": str, "id_vehicule": str}
)

num_acc_absents = ~df_usagers_concat["Num_Acc"].astype(str).isin(
    df_caract_clean["Num_Acc"]
)

id_vehicule_absents = ~df_usagers_concat["id_vehicule"].astype(str).isin(
    df_vehicules_clean["id_vehicule"]
)

print("Usagers sans accident correspondant :", num_acc_absents.sum())
print("Usagers sans véhicule correspondant :", id_vehicule_absents.sum())

Usagers sans accident correspondant : 0
Usagers sans véhicule correspondant : 0


#### Résultat

Tous les usagers possèdent un `Num_Acc` correspondant à un accident présent dans la table Caractéristiques.

De même, tous les `id_vehicule` de la table Usagers correspondent à un véhicule présent dans la table Véhicules.

Aucune ligne orpheline n'est donc détectée. Les relations entre les tables Usagers, Caractéristiques et Véhicules sont cohérentes et pourront être utilisées lors de la modélisation relationnelle.

In [481]:
# 2.8. Contrôle des types

print(df_usagers_concat.dtypes)

Num_Acc          int64
id_usager       object
id_vehicule        str
place            int64
catu             int64
grav             int64
sexe             int64
an_nais        float64
trajet           int64
secu1            int64
dtype: object


#### Résultat

Les types obtenus après consolidation sont globalement cohérents, mais certaines conversions seront nécessaires lors du nettoyage.

Les identifiants `Num_Acc`, `id_usager` et `id_vehicule` seront traités comme des chaînes de caractères.

Les variables codées `place`, `catu`, `grav`, `sexe`, `trajet` et `secu1` seront converties au format entier nullable `Int64` afin de permettre la présence de valeurs manquantes.

La variable `an_nais`, actuellement en `float64` en raison de valeurs manquantes, sera également convertie en `Int64`.

### 2.9. Synthèse des décisions de nettoyage

À l'issue des contrôles qualité, les décisions suivantes sont retenues :

| Variable | Constat | Décision |
|---|---|---|
| `Num_Acc` | Aucun identifiant manquant | Conserver et convertir en chaîne de caractères |
| `id_usager` | Absent en 2020 (105 295 valeurs manquantes) | Conserver comme identifiant source BAAC, avec les valeurs manquantes de 2020 |
| `id_vehicule` | Aucun identifiant manquant | Conserver et convertir en chaîne de caractères |
| `place` | 30 codes `-1` | Remplacer `-1` par une valeur manquante |
| `catu` | Aucun code `-1` détecté | Conserver sans traitement des valeurs |
| `grav` | 419 codes `-1` | Remplacer `-1` par une valeur manquante |
| `sexe` | 10 632 codes `-1` | Remplacer `-1` par une valeur manquante |
| `an_nais` | 11 118 valeurs manquantes | Conserver les valeurs manquantes et convertir en `Int64` |
| `trajet` | 11 482 codes `-1` et 160 424 codes `0` | Remplacer `-1` et `0` par des valeurs manquantes |
| `secu1` | 9 898 codes `-1` ; le code `0` est une modalité valide | Remplacer uniquement `-1` par une valeur manquante |

Un identifiant technique `id_usager_projet` sera créé pour l'ensemble des 612 181 lignes. Il permettra d'identifier de manière unique chaque enregistrement et pourra servir de clé primaire de la table Usagers lors de la modélisation relationnelle.

L'identifiant `id_usager` fourni par BAAC sera conservé séparément afin de préserver l'information source. Les valeurs manquantes de 2020 ne seront pas remplacées par des identifiants BAAC artificiels.

Aucun doublon n'est supprimé. Les 65 doublons complets détectés concernent uniquement 2020, année pour laquelle `id_usager` n'est pas disponible, et il n'est pas possible d'établir qu'ils correspondent à des usagers réellement dupliqués.

Aucune ligne orpheline n'a été détectée : tous les `Num_Acc` correspondent à la table Caractéristiques et tous les `id_vehicule` correspondent à la table Véhicules.

Aucune imputation ne sera réalisée dans le fichier de référence nettoyé. Les traitements nécessaires au machine learning seront réalisés ultérieurement dans le pipeline de préparation du modèle.

La variable `an_nais` sera conservée puis renommée `annee_naissance`. Une variable `age` sera créée ultérieurement dans le jeu de données analytique après rapprochement avec l'année de l'accident.

## 3. Nettoyage, renommage, validation et export

In [482]:
# 3.1. Création du DataFrame de nettoyage

df_usagers_clean = df_usagers_concat.copy()

print("Dimensions :", df_usagers_clean.shape)

Dimensions : (612181, 10)


In [483]:
# 3.2. Traitement des codes non renseignés

colonnes_moins_un = [
    "place",
    "grav",
    "sexe",
    "secu1"
]

for colonne in colonnes_moins_un:
    df_usagers_clean[colonne] = df_usagers_clean[colonne].replace(-1, pd.NA)

df_usagers_clean["trajet"] = df_usagers_clean["trajet"].replace(
    [-1, 0],
    pd.NA
)

In [484]:
# 3.2.1. Vérification du traitement

for colonne in ["place", "grav", "sexe", "secu1"]:
    print(f"{colonne} - codes -1 restants : {(df_usagers_clean[colonne] == -1).sum()}")

print(
    "trajet - codes -1 restants :",
    (df_usagers_clean["trajet"] == -1).sum()
)

print(
    "trajet - codes 0 restants :",
    (df_usagers_clean["trajet"] == 0).sum()
)

place - codes -1 restants : 0
grav - codes -1 restants : 0
sexe - codes -1 restants : 0
secu1 - codes -1 restants : 0
trajet - codes -1 restants : 0
trajet - codes 0 restants : 0


In [485]:
# 3.3. Conversion des types

df_usagers_clean["Num_Acc"] = df_usagers_clean["Num_Acc"].astype("string")
df_usagers_clean["id_usager"] = df_usagers_clean["id_usager"].astype("string")
df_usagers_clean["id_vehicule"] = df_usagers_clean["id_vehicule"].astype("string")

colonnes_entieres = [
    "place",
    "catu",
    "grav",
    "sexe",
    "an_nais",
    "trajet",
    "secu1"
]

for colonne in colonnes_entieres:
    df_usagers_clean[colonne] = df_usagers_clean[colonne].astype("Int64")

In [486]:
# 3.3.1. Vérification des types

print(df_usagers_clean.dtypes)

Num_Acc        string
id_usager      string
id_vehicule    string
place           Int64
catu            Int64
grav            Int64
sexe            Int64
an_nais         Int64
trajet          Int64
secu1           Int64
dtype: object


In [487]:
# 3.4. Création de l'identifiant technique des usagers

df_usagers_clean.insert(
    0,
    "id_usager_projet",
    range(1, len(df_usagers_clean) + 1)
)

df_usagers_clean["id_usager_projet"] = (
    df_usagers_clean["id_usager_projet"].astype("Int64")
)

print(df_usagers_clean[["id_usager_projet", "id_usager"]].head())

   id_usager_projet id_usager
0                 1      <NA>
1                 2      <NA>
2                 3      <NA>
3                 4      <NA>
4                 5      <NA>


In [488]:
# 3.4.1. Vérification de l'identifiant technique

print("Valeurs manquantes :", df_usagers_clean["id_usager_projet"].isna().sum())
print("Doublons :", df_usagers_clean["id_usager_projet"].duplicated().sum())
print("Nombre d'identifiants uniques :", df_usagers_clean["id_usager_projet"].nunique())

Valeurs manquantes : 0
Doublons : 0
Nombre d'identifiants uniques : 612181


In [489]:
# 3.5. Renommage des variables

df_usagers_clean = df_usagers_clean.rename(
    columns={
        "place": "place_usager",
        "catu": "categorie_usager",
        "grav": "gravite",
        "an_nais": "annee_naissance",
        "trajet": "motif_trajet",
        "secu1": "equipement_securite"
    }
)

print(df_usagers_clean.columns.tolist())

['id_usager_projet', 'Num_Acc', 'id_usager', 'id_vehicule', 'place_usager', 'categorie_usager', 'gravite', 'sexe', 'annee_naissance', 'motif_trajet', 'equipement_securite']


In [490]:
# 3.6. Validation finale

print("Dimensions :", df_usagers_clean.shape)
print()

print("Valeurs manquantes :")
print(df_usagers_clean.isna().sum())
print()

print("Doublons sur id_usager_projet :", df_usagers_clean["id_usager_projet"].duplicated().sum())
print("Valeurs manquantes sur id_usager_projet :", df_usagers_clean["id_usager_projet"].isna().sum())

Dimensions : (612181, 11)

Valeurs manquantes :
id_usager_projet            0
Num_Acc                     0
id_usager              105295
id_vehicule                 0
place_usager               30
categorie_usager            0
gravite                   419
sexe                    10632
annee_naissance         11118
motif_trajet           171906
equipement_securite      9898
dtype: int64

Doublons sur id_usager_projet : 0
Valeurs manquantes sur id_usager_projet : 0


#### Résultat

Le jeu de données nettoyé contient 612 181 usagers et 11 variables.

Les valeurs manquantes restantes correspondent aux absences d'information identifiées et documentées lors du contrôle qualité. Aucune imputation n'a été réalisée dans le fichier de référence.

L'identifiant technique `id_usager_projet` est renseigné et unique pour chaque usager. Les identifiants `Num_Acc` et `id_vehicule` sont également entièrement renseignés.

Le jeu de données est désormais prêt pour l'export dans le dossier `processed`.

In [491]:
# 3.7. Export des données nettoyées

df_usagers_clean.to_csv(
    "data/processed/usagers_2020_2024_clean.csv",
    sep=";",
    index=False
)

print("Fichier nettoyé sauvegardé.")

Fichier nettoyé sauvegardé.


### Conclusion

Les données Usagers 2020-2024 ont été harmonisées, contrôlées et nettoyées.

Le fichier final contient 612 181 usagers et 11 variables. Les valeurs non renseignées ont été traitées selon les règles définies lors du contrôle qualité, sans imputation.

Un identifiant technique unique `id_usager_projet` a été créé pour chaque enregistrement afin de préparer la future modélisation relationnelle.

Le fichier nettoyé est disponible dans `data/processed/usagers_2020_2024_clean.csv`.